In [ ]:
!pip install git+https://github.com/facebookresearch/fairchem.git@fairchem_core-2.0.0#subdirectory=packages/fairchem-core

In [2]:
# colab
!git clone https://github.com/yasheshak/Chem-277B-Final-Project.git

fatal: destination path 'Chem-277B-Final-Project' already exists and is not an empty directory.


In [3]:
%cd Chem-277B-Final-Project

/content/Chem-277B-Final-Project


In [4]:
!ls -a

.			    environment.yml	      README.md
..			    extract.py		      read_multi_ase.py
Chem-277B-Final-Project     .git		      SchNet_GNN_Baseline.ipynb
db_explore.ipynb	    hyperparam_test.ipynb     SchNet_GNN.ipynb
DimeNet_baseline_new.ipynb  hyperparam_test_v2.ipynb  SchNet_Normalize.ipynb
EDA			    Makefile		      simpleGNN
EDA.ipynb		    __pycache__


Goal: Optimizer hyperparameters

Plan
1. Baseline sweep: identify how sensitive the model is when changing target parameter. A range of values for a single parameter is tested while keeping the other params at its default value.
    - Narrow down hyperparameter values
2. Grid search: explores interactions between hyperparameters by examining every possible combination.

Baseline sweep
1. create dict of start, stop, min (user defined)


In [5]:
import torch
print(torch.__version__)

2.6.0+cu124


In [6]:
import sys
sys.path.append('/content/Chem-277B-Final-Project')
from read_multi_ase import *
from extract import *
from SchNet_for_import import *

import numpy as np
import glob
from typing import Union
import matplotlib.pyplot as plt
# import torch
from torch.utils.data import random_split
from torch_geometric.data import Data
from torch_geometric.nn import SchNet
from torch_geometric.loader import DataLoader
from fairchem.core.datasets import AseDBDataset
from torch_cluster import radius_graph

/usr/local/lib/python3.12/dist-packages/torch_geometric/typing.py:84: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /usr/local/lib/python3.12/dist-packages/libpyg.so: undefined symbol: _ZN3c1010Dispatcher17runRecordFunctionERN2at14RecordFunctionESt17reference_wrapperIKNS_14FunctionSchemaEENS_11DispatchKeyE
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/usr/local/lib/python3.12/dist-packages/torch_geometric/typing.py:128: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /usr/local/lib/python3.12/dist-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSs
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/usr/local/lib/python3.12/dist-packages/torch_geometric/typing.py:139: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: /usr/local/lib/python3.12/dist-packages/torch_c

ModuleNotFoundError: No module named 'SchNet_for_import'

In [9]:
from google.colab import auth
auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [37]:
# ---------------------------------- Initialize hyperparam range -------------------------------
# parameters to optimize
hidden_channels = {
    'start': 1,
    'end': 1,
    'step': 1
}
# num_filters
num_filters = {
    'start': 1,
    'end': 1,
    'step': 1
}
# num_interactions
num_interactions = {
    'start': 1,
    'end': 1,
    'step': 1
}
# num_gaussians
num_gaussians = {
    'start': 1,
    'end': 1,
    'step': 1
}
# cutoff
cutoff = {
    'start': 1,
    'end': 1,
    'step': 1
}
# max_num_neighbors
max_num_neighbors = {
    'start': 1,
    'end': 1,
    'step': 1
}

params_to_optimize = [hidden_channels, num_filters, num_interactions, num_gaussians, cutoff, max_num_neighbors]

# set parameters
# set_params = [readout, dipole, mean, std, atomref, train_mean, train_std]

In [38]:
# ------------------------------ Initialize training set --------------------------------
# given the local dataset path, loads the first .aselmdb file
dataset_path = '/content/drive/MyDrive/train_4M/data0000.aselmdb'
# dataset = AseDBDataset({"src": dataset_path})
files_list = dataset_path
dataset = process_file(files_list, molecule_type = 'biomolecules', max_molecules = 200)

# convert to torch
torch_data = get_data(dataset)
train_dataset, val_dataset = split_data(torch_data, 0.8)

# load
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

Processed 200 atoms


In [39]:
# ------------------------------- Final Function -------------------------------------------
def param_tuning(param: str, param_start: int, param_end: int, param_step: int):
    # initialize numpy array for param vals
    param_vals = np.arange(start=param_start, stop=(param_end+1), step=param_step)

    # initialize default params
    default_params = {
    'hidden_channels': 128,
    'num_filters': 128,
    'num_interactions': 6,
    'num_gaussians': 50,
    'cutoff': 5,
    'max_num_neighbors': 32
    }

    # update default_params
    default_readout = 'add'
    default_dipole = False
    default_mean = None
    default_std = None
    default_atomref = None
    default_train_mean = None
    default_train_std = None

    for val in param_vals:
      # update param dict
      default_params[param] = val

      # initialize model
      device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
      #Initialize model with desired parameters
      model = SchNetModel(
          default_params['hidden_channels'],
          default_params['num_filters'],
          default_params['num_interactions'],
          default_params['num_gaussians'],
          default_params['cutoff'],
          default_params['max_num_neighbors'],
          readout = "add",
          dipole = False,
          mean = None,
          std = None,
          atomref = None,
          train_mean = None,
          train_std = None
      ).to(device)

      optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
      loss_function = torch.nn.SmoothL1Loss()

      # run model
      epochs = 50
      train_losses = np.zeros(epochs)
      val_losses = np.zeros(epochs)

      for epoch in range(50):
          train_loss = train(model, train_loader)
          val_loss = evaluate(model, val_loader)

          train_losses[epoch] = train_loss
          val_losses[epoch] = val_loss

      print(f'hyperparameter: {param}, value: {val}')

      # save photo of RMSE graph
      # plot_losses(bio_train_losses, bio_val_losses)
      # save graph

      # save combos to df then excel (maybe graph?)

In [40]:
# ================================== Run Baseline Sweep =================================

param_tuning('num_interactions', 1, 2, 1)

ImportError: 'radius_graph' requires 'torch-cluster'